# CSV1 剩余均价与清算标记价复核
仅复核v0.6.0的166笔主均价空值与3笔原始标记价空值。原始数据在本地查证；本笔记离线验证交付文件及证据，不声称重新取得全部历史原档。


In [1]:
from pathlib import Path
import csv,hashlib,json,sys
candidates=[Path.cwd(),Path.cwd()/'results/csv1_full_20250322_20261031']
for folder in candidates:
    public=folder/'csv1_20250322_20261031_victim_prices_precision_zh.csv'
    local=folder/'csv1_20250322_20261031_含account_value_开仓价清算价_数量精度复核.csv'
    if public.exists():
        output=public;baseline=folder/'csv1_20250322_20261031_victim_prices_zh.csv'
        module_path=folder.parents[1]/'tools/validate_csv1_victim_precision.py';root=folder.parents[1];break
    if local.exists():
        output=local;baseline=folder/'baseline_v060.csv'
        module_path=folder/'validate_precision_results.py';root=folder;break
else:raise FileNotFoundError('Run from repository root or result folder')
sys.path.insert(0,str(root))
import importlib.util
spec=importlib.util.spec_from_file_location('precision_validator',module_path)
validator=importlib.util.module_from_spec(spec);spec.loader.exec_module(validator)
evidence=json.loads((folder/'precision_validation.json').read_text(encoding='utf-8'))
assert hashlib.sha256(output.read_bytes()).hexdigest()==evidence['sha256']
print('CSV bytes/SHA256 and portable paths: PASS')


CSV bytes/SHA256 and portable paths: PASS


In [2]:
actual=validator.check(output,baseline,folder/'precision_event_audit.json')
for key in ['sha256','bytes','columns','data_rows','counts','changed_cells']:
    assert actual[key]==evidence[key],key
print('原46列逐格一致；仅166/3目标列获准变更；其他58列基线单元格不变')


原46列逐格一致；仅166/3目标列获准变更；其他58列基线单元格不变


In [3]:
audit=json.loads((folder/'precision_event_audit.json').read_text(encoding='utf-8'))
assert len(audit['details'])==166 and len(audit['missing_mark_details'])==3
print(json.dumps(actual['counts'],ensure_ascii=False,indent=2))
assert actual['counts']['newly_reconstructed']+actual['counts']['newly_from_archive_state']+actual['counts']['remaining_entry']==166
assert actual['counts']['mark_filled']==22041


{
  "fills": 22044,
  "average_entry_filled": 22040,
  "mark_filled": 22041,
  "newly_filled_entry": 162,
  "newly_reconstructed": 111,
  "newly_from_archive_state": 51,
  "local_unresolved_position_chain_discontinuity": 4,
  "local_unresolved_source_time_coverage_incomplete": 44,
  "events_with_tolerated_tails": 15,
  "recovered_with_tolerated_tails": 8,
  "recovery_requires_quantity_tolerance": 8,
  "local_unresolved_no_zero_or_reversal_anchor": 6,
  "remaining_entry": 4,
  "remaining_archive_entry_disagrees_with_native_close_pnl": 2,
  "remaining_archive_entry_unavailable": 2,
  "local_unresolved_ambiguous_same_millisecond_order": 1,
  "diagnostic_rows": 4
}


In [4]:
from decimal import Decimal
from victim_quantity_precision import QuantityTailPolicy
p=QuantityTailPolicy(5,Decimal('0.00001'))
assert p.equal('67858660.19999999','67858660.20000000')
assert not p.equal('0.1','0.10001')
for r in audit['details']:
    if r['local_reason']=='position_chain_discontinuity':
        b=r['blocker'];assert abs(Decimal(b['difference']))>Decimal(b['tolerance'])
    if r['recovery_method']=='local_history':
        assert r['source_time_coverage']['covered'] and not r['source_time_coverage']['gaps']
    elif r['recovery_method']=='third_party_pre_fill_state':
        proof=r['archive_verification']['proof']
        assert proof['no_future_entry_state'] and proof['target_identity_and_financial_fields_verified']
        assert Decimal(proof['pnl_abs_difference_usdc'])<=Decimal(proof['pnl_validation_bound_usdc'])
    else:assert r['remaining_reason_zh']
print('用户尾差示例通过；一个真实最小下单单位仍不等；所有剩余断点超过各自容限')


用户尾差示例通过；一个真实最小下单单位仍不等；所有剩余断点超过各自容限


In [5]:
for m in audit['missing_mark_details']:
    assert m['fresh_official_query']['http_status']==200 and m['fresh_official_query']['records']==0
    print(m['coin'],m['event_id'],m['reason_zh'])
assert evidence['aws_requests']==0 and evidence['cash_spent_usd']==0
print('3笔缺失原始markPx逐笔保留原因；独立账本参考未冒充原始字段；无AWS请求')


TRUMP bd5f9795b0ce267c203ad5f3b69495bc7fb7b7923eefd5eb43ea09ef5260053d 原始 node_trades node_trades/hourly/20250423/16.lz4:119316（行SHA256 7b43ab8cbd747706ea27bfb539324b7eeedc1789dfb49702b6f32d90c3b605b0）只有成交价/数量，没有 markPx；本轮官方 userFillsByTime 分别查询被接管用户与实际接收方HLP账户的目标时刻前后2秒，双方均返回0笔，因此不是数量尾差筛选阻断；官方 txDetails/区块 572211361 的 blockDetails 均无该标记价；Hypedexer 已下载当日299笔CSV，但没有目标hash成交，也没有 markPx 字段；本轮使用现有完整历史权限重查0xArchive，清算接口返回400/range_before_coverage（该数据集仅服务2025-12-22起的清算），历史仓位接口返回0仓位并明确注明coverage_from=2025-05-25 15:00 UTC，均晚于本事件；确认是数据集覆盖起始时间限制；现有单仓位账本隐含值仅作独立参考，不冒充原始标记价
kPEPE a9dc9897d8d9f4b9e3bb017af85dcc2280d9707384509719f027a9e7dfef221f 原始 node_trades node_trades/hourly/20250509/8.lz4:146973（行SHA256 8102fb6762894a6e462a6e4a36cceca35ad016b0b36a33e8e92f171226cfcf19）只有成交价/数量，没有 markPx；本轮官方 userFillsByTime 分别查询被接管用户与实际接收方HLP账户的目标时刻前后2秒，双方均返回0笔，因此不是数量尾差筛选阻断；官方 txDetails/区块 588967704 的 blockDetails 均无该标记价；本轮Hypedexer提示每IP每24小时1次导出限制，未取得文件；本轮使用现有完整历史权限重查0xArchive，清算接口返回400/range_before_coverage（该数据集